# Bank Churners — BI Reporting

**Objective:** Produce BI-ready tables, figures, and an executive summary for retention planning.

**Data sources:** `data/raw/BankChurners.csv` (observed churn) and `artifacts/models/churn_model_pipeline.joblib` (model scores).

**Important distinctions**
- *Observed churn* comes from `Attrition_Flag` in the historical portfolio.
- *Model-based risk* comes from predicted churn probability on the same customers.
- Correlation patterns are descriptive only (not causal claims).

**Outputs:** `reports/tables/` (including **`power_bi_master.csv`** for Power BI), `reports/figures/`, `reports/executive_summary.md`


In [1]:
# =============================================================================
# 0. Setup — paths, constants, helpers
# =============================================================================
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "BankChurners.csv").exists():
    ROOT = Path.cwd().parent
if not (ROOT / "data" / "raw" / "BankChurners.csv").exists():
    raise FileNotFoundError("Place BankChurners.csv in data/raw/")

RAW_PATH = ROOT / "data" / "raw" / "BankChurners.csv"
MODEL_PATH = ROOT / "artifacts" / "models" / "churn_model_pipeline.joblib"
FIG_DIR = ROOT / "reports" / "figures"
TABLE_DIR = ROOT / "reports" / "tables"
EXEC_SUMMARY_PATH = ROOT / "reports" / "executive_summary.md"
MODEL_METRICS_PATH = TABLE_DIR / "modeling_final_metrics.csv"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

LEAKAGE_COLS = [
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2",
]
TARGET_COL = "Attrition_Flag"
ID_COL = "CLIENTNUM"

# Default operational threshold; override if modeling exported a threshold file.
HIGH_RISK_THRESHOLD = 0.5
threshold_path = ROOT / "artifacts" / "logs" / "decision_threshold.json"
if threshold_path.exists():
    import json as _json

    with open(threshold_path, encoding="utf-8") as f:
        HIGH_RISK_THRESHOLD = float(_json.load(f).get("threshold", HIGH_RISK_THRESHOLD))

CHURN_COLOR = "#0072B2"
NON_CHURN_COLOR = "#E69F00"
PALETTE = [NON_CHURN_COLOR, CHURN_COLOR]

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight"})


def assign_bands(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["age_band"] = pd.cut(
        out["Customer_Age"],
        bins=[0, 34, 49, 64, 120],
        labels=["Under 35", "35-49", "50-64", "65+"],
        right=True,
    )
    out["tenure_band"] = pd.cut(
        out["Months_on_book"],
        bins=[-1, 24, 48, 72, 10_000],
        labels=["0-24 mo", "25-48 mo", "49-72 mo", "73+ mo"],
        right=True,
    )
    out["inactivity_band"] = pd.cut(
        out["Months_Inactive_12_mon"],
        bins=[-1, 1, 3, 6, 100],
        labels=["0-1 mo", "2-3 mo", "4-6 mo", "6+ mo"],
        right=True,
    )
    return out


def segment_churn_table(
    frame: pd.DataFrame,
    dimension: str,
    value_col: str,
) -> pd.DataFrame:
    grouped = (
        frame.groupby(value_col, observed=False)
        .agg(
            customer_count=(ID_COL, "count"),
            attrited_count=("is_churn", "sum"),
            observed_churn_rate=("is_churn", "mean"),
            avg_churn_probability=("churn_probability", "mean"),
            predicted_high_risk_count=("predicted_churn", "sum"),
        )
        .reset_index()
    )
    grouped = grouped.rename(columns={value_col: "segment_value"})
    grouped.insert(0, "segment_dimension", dimension)
    grouped["observed_churn_rate_pct"] = (grouped["observed_churn_rate"] * 100).round(2)
    grouped["avg_churn_probability_pct"] = (grouped["avg_churn_probability"] * 100).round(2)
    return grouped


In [2]:
# =============================================================================
# 1. Load raw data and trained pipeline
# =============================================================================
df_raw = pd.read_csv(RAW_PATH)
print("Portfolio shape:", df_raw.shape)

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing model artifact at {MODEL_PATH}. Train and save the pipeline in notebooks/02_modeling.ipynb."
    )

artifact = joblib.load(MODEL_PATH)
if isinstance(artifact, dict):
    pipeline = artifact["pipeline"]
    HIGH_RISK_THRESHOLD = float(
        artifact.get("threshold", artifact.get("final_threshold", HIGH_RISK_THRESHOLD))
    )
    meta = artifact.get("metadata", {})
    FINAL_MODEL_NAME = meta.get("model_name", "unknown")
    FEATURE_COLUMNS = list(meta.get("feature_columns", []))
else:
    pipeline = artifact
    FINAL_MODEL_NAME = type(pipeline).__name__
    FEATURE_COLUMNS = list(getattr(pipeline, "feature_names_in_", []))

print("Loaded model:", FINAL_MODEL_NAME, "| threshold:", HIGH_RISK_THRESHOLD)


Portfolio shape: (10127, 23)
Loaded model: lightgbm | threshold: 0.5


In [3]:
# =============================================================================
# 2. Score full portfolio
# =============================================================================
df = assign_bands(df_raw.copy())
df["is_churn"] = (df[TARGET_COL] == "Attrited Customer").astype(int)

if not FEATURE_COLUMNS:
    drop_cols = {TARGET_COL, ID_COL, *LEAKAGE_COLS}
    FEATURE_COLUMNS = [c for c in df.columns if c not in drop_cols]

missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"Model expects columns missing from raw data: {missing}")
feature_df = df[FEATURE_COLUMNS]

if hasattr(pipeline, "predict_proba"):
    churn_probability = pipeline.predict_proba(feature_df)[:, 1]
else:
    churn_probability = pipeline.decision_function(feature_df)
    churn_probability = (churn_probability - churn_probability.min()) / (
        churn_probability.max() - churn_probability.min() + 1e-9
    )

df["churn_probability"] = churn_probability
df["predicted_churn"] = (df["churn_probability"] >= HIGH_RISK_THRESHOLD).astype(int)
df["is_high_risk"] = df["predicted_churn"].astype(bool)

print(
    f"Scored {len(df):,} customers | high-risk threshold = {HIGH_RISK_THRESHOLD:.3f} | "
    f"high-risk count = {df['is_high_risk'].sum():,}"
)
df[[ID_COL, TARGET_COL, "churn_probability", "predicted_churn", "Card_Category", "Income_Category"]].head()


Scored 10,127 customers | high-risk threshold = 0.500 | high-risk count = 1,594


,CLIENTNUM,Attrition_Flag,churn_probability,predicted_churn,Card_Category,Income_Category
0,768805383,Existing Customer,0.000015,0,Blue,$60K - $80K
1,818770008,Existing Customer,0.000019,0,Blue,Less than $40K
2,713982108,Existing Customer,0.000033,0,Blue,$80K - $120K
3,769911858,Existing Customer,0.000470,0,Blue,Less than $40K
4,709106358,Existing Customer,0.207069,0,Blue,$60K - $80K


In [4]:
# =============================================================================
# 3. KPI table — overall churn rate summary
# =============================================================================
total_customers = len(df)
observed_attrited = int(df["is_churn"].sum())
observed_churn_rate = df["is_churn"].mean()
predicted_high_risk = int(df["is_high_risk"].sum())
predicted_high_risk_rate = df["is_high_risk"].mean()
avg_model_score = df["churn_probability"].mean()

churn_rate_summary = pd.DataFrame(
    [
        {
            "metric": "total_customers",
            "value": total_customers,
            "notes": "Full portfolio in raw file",
        },
        {
            "metric": "observed_attrited_customers",
            "value": observed_attrited,
            "notes": "Attrition_Flag = Attrited Customer",
        },
        {
            "metric": "observed_churn_rate_pct",
            "value": round(observed_churn_rate * 100, 2),
            "notes": "Descriptive historical churn rate",
        },
        {
            "metric": "avg_model_churn_probability_pct",
            "value": round(avg_model_score * 100, 2),
            "notes": "Mean predicted churn probability (model-based)",
        },
        {
            "metric": "high_risk_customers_model",
            "value": predicted_high_risk,
            "notes": f"Predicted churn at probability >= {HIGH_RISK_THRESHOLD:.3f}",
        },
        {
            "metric": "high_risk_rate_pct",
            "value": round(predicted_high_risk_rate * 100, 2),
            "notes": "Share of portfolio flagged for follow-up",
        },
    ]
)

churn_rate_path = TABLE_DIR / "churn_rate_summary.csv"
churn_rate_summary.to_csv(churn_rate_path, index=False)
print("Saved:", churn_rate_path)
churn_rate_summary


Saved: C:\Users\Windows\Documents\Repo Folder\bank-churners-cursor-bootcamp\reports\tables\churn_rate_summary.csv


,metric,value,notes
0,total_customers,10127.00,Full portfolio in raw file
1,observed_attrited_customers,1627.00,Attrition_Flag = Attrited Customer
2,observed_churn_rate_pct,16.07,Descriptive historical churn rate
3,avg_model_churn_probability_pct,15.82,Mean predicted churn probability (model-based)
4,high_risk_customers_model,1594.00,Predicted churn at probability >= 0.500
5,high_risk_rate_pct,15.74,Share of portfolio flagged for follow-up


In [5]:
# =============================================================================
# 4. Segment-level churn analysis
# =============================================================================
segment_specs = [
    ("card_category", "Card_Category"),
    ("income_category", "Income_Category"),
    ("age_band", "age_band"),
    ("tenure_band", "tenure_band"),
    ("inactivity_band", "inactivity_band"),
]

segment_tables = [
    segment_churn_table(df, dimension, col) for dimension, col in segment_specs
]
churn_by_segment = pd.concat(segment_tables, ignore_index=True)
churn_by_segment = churn_by_segment.sort_values(
    ["segment_dimension", "observed_churn_rate"],
    ascending=[True, False],
)

segment_path = TABLE_DIR / "churn_by_segment.csv"
churn_by_segment.to_csv(segment_path, index=False)
print("Saved:", segment_path)
churn_by_segment.head(10)


Saved: C:\Users\Windows\Documents\Repo Folder\bank-churners-cursor-bootcamp\reports\tables\churn_by_segment.csv


,segment_dimension,segment_value,customer_count,attrited_count,observed_churn_rate,avg_churn_probability,predicted_high_risk_count,observed_churn_rate_pct,avg_churn_probability_pct
12,age_band,50-64,3419,566,0.165545,0.162072,552,16.55,16.21
11,age_band,35-49,5862,949,0.161890,0.160415,935,16.19,16.04
10,age_band,Under 35,735,101,0.137415,0.132593,97,13.74,13.26
13,age_band,65+,111,11,0.099099,0.094358,10,9.91,9.44
2,card_category,Platinum,20,5,0.250000,0.249590,5,25.00,24.96
1,card_category,Gold,116,21,0.181034,0.189100,22,18.10,18.91
0,card_category,Blue,9436,1519,0.160979,0.158370,1486,16.10,15.84
3,card_category,Silver,555,82,0.147748,0.146129,81,14.77,14.61
20,inactivity_band,4-6 mo,737,181,0.245590,0.237472,176,24.56,23.75
19,inactivity_band,2-3 mo,7128,1331,0.186728,0.185156,1313,18.67,18.52


In [6]:
# =============================================================================
# 5. Model performance summary (if modeling metrics exist)
# =============================================================================
if MODEL_METRICS_PATH.exists():
    model_metrics = pd.read_csv(MODEL_METRICS_PATH)
    test_metrics = model_metrics.loc[model_metrics["split"] == "test"].copy()
    if test_metrics.empty:
        test_metrics = model_metrics.copy()
    model_metrics_summary = test_metrics.rename(
        columns={
            "roc_auc": "roc_auc",
            "precision": "precision_churn_class",
            "recall": "recall_churn_class",
            "f1": "f1_churn_class",
            "accuracy": "accuracy",
            "model": "model_name",
        }
    )
else:
    model_metrics_summary = pd.DataFrame(
        [
            {
                "model_name": "unavailable",
                "split": "n/a",
                "roc_auc": np.nan,
                "precision_churn_class": np.nan,
                "recall_churn_class": np.nan,
                "f1_churn_class": np.nan,
                "accuracy": np.nan,
                "notes": "Run notebooks/02_modeling.ipynb to create modeling_final_metrics.csv",
            }
        ]
    )

metrics_path = TABLE_DIR / "model_metrics_summary.csv"
model_metrics_summary.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)
model_metrics_summary


Saved: C:\Users\Windows\Documents\Repo Folder\bank-churners-cursor-bootcamp\reports\tables\model_metrics_summary.csv


,model_name,split,roc_auc,precision_churn_class,recall_churn_class,f1_churn_class,accuracy,threshold
2,lightgbm,test,0.99163,0.920705,0.856557,0.887473,0.965132,0.5


In [7]:
# =============================================================================
# 6. High-risk customer export for operations
# =============================================================================
high_risk_cols = [
    ID_COL,
    "churn_probability",
    "predicted_churn",
    "Card_Category",
    "Income_Category",
    "age_band",
    "tenure_band",
    "inactivity_band",
    "Months_on_book",
    "Months_Inactive_12_mon",
    "Total_Trans_Ct",
    "Contacts_Count_12_mon",
]
high_risk_customers = (
    df.loc[df["is_high_risk"], high_risk_cols]
    .sort_values("churn_probability", ascending=False)
    .rename(
        columns={
            "churn_probability": "churn_probability_score",
            "predicted_churn": "predicted_churn_class",
        }
    )
)
high_risk_customers.insert(0, "high_risk_threshold_used", HIGH_RISK_THRESHOLD)

high_risk_path = TABLE_DIR / "high_risk_customers.csv"
high_risk_customers.to_csv(high_risk_path, index=False)
print("Saved:", high_risk_path, "| rows:", len(high_risk_customers))
high_risk_customers.head()


Saved: C:\Users\Windows\Documents\Repo Folder\bank-churners-cursor-bootcamp\reports\tables\high_risk_customers.csv | rows: 1594


,high_risk_threshold_used,CLIENTNUM,churn_probability_score,predicted_churn_class,Card_Category,Income_Category,age_band,tenure_band,inactivity_band,Months_on_book,Months_Inactive_12_mon,Total_Trans_Ct,Contacts_Count_12_mon
9602,0.5,715645908,0.999984,1,Gold,$80K - $120K,50-64,25-48 mo,2-3 mo,48,2,71,6
8379,0.5,779183958,0.999982,1,Blue,Less than $40K,35-49,25-48 mo,2-3 mo,33,3,38,4
10018,0.5,709566408,0.999981,1,Blue,Less than $40K,35-49,25-48 mo,2-3 mo,30,3,70,6
564,0.5,752604633,0.999976,1,Blue,$40K - $60K,50-64,49-72 mo,4-6 mo,50,5,17,3
6244,0.5,718653333,0.999973,1,Blue,Less than $40K,35-49,25-48 mo,2-3 mo,33,3,31,3


In [8]:
# =============================================================================
# 6b. Power BI master table (single CSV for beginners)
# =============================================================================
# One row per customer: observed churn + model scores + segmentation bands.
# Import only this file in Power BI to build KPI cards, segment bars, and risk tables.

def assign_risk_band(probability: float) -> str:
    if probability >= HIGH_RISK_THRESHOLD:
        return "High"
    if probability >= 0.70:
        return "Elevated"
    if probability >= 0.35:
        return "Medium"
    return "Low"


master_columns = [
    ID_COL,
    TARGET_COL,
    "observed_churn",
    "churn_probability",
    "predicted_churn",
    "is_high_risk",
    "risk_band",
    "Gender",
    "Education_Level",
    "Marital_Status",
    "Income_Category",
    "Card_Category",
    "age_band",
    "tenure_band",
    "inactivity_band",
    "Customer_Age",
    "Dependent_count",
    "Months_on_book",
    "Months_Inactive_12_mon",
    "Contacts_Count_12_mon",
    "Credit_Limit",
    "Total_Revolving_Bal",
    "Avg_Open_To_Buy",
    "Total_Trans_Amt",
    "Total_Trans_Ct",
    "Total_Amt_Chng_Q4_Q1",
    "Total_Ct_Chng_Q4_Q1",
    "Avg_Utilization_Ratio",
    "Total_Relationship_Count",
    "decision_threshold_used",
    "model_name",
]

power_bi_master = df.copy()
power_bi_master["observed_churn"] = power_bi_master["is_churn"].astype(int)
power_bi_master["risk_band"] = power_bi_master["churn_probability"].apply(assign_risk_band)
power_bi_master["decision_threshold_used"] = HIGH_RISK_THRESHOLD
power_bi_master["model_name"] = FINAL_MODEL_NAME

# Keep bands as plain strings for Power BI (avoid categorical dtype issues)
for band_col in ["age_band", "tenure_band", "inactivity_band"]:
    power_bi_master[band_col] = power_bi_master[band_col].astype(str)

power_bi_master = power_bi_master[master_columns]
master_path = TABLE_DIR / "power_bi_master.csv"
power_bi_master.to_csv(master_path, index=False)

print("Saved:", master_path)
print("Rows:", len(power_bi_master), "| Columns:", len(power_bi_master.columns))
print("Tip: In Power BI → Get Data → Text/CSV → select power_bi_master.csv")
power_bi_master.head()

Saved: C:\Users\Windows\Documents\Repo Folder\bank-churners-cursor-bootcamp\reports\tables\power_bi_master.csv
Rows: 10127 | Columns: 31
Tip: In Power BI → Get Data → Text/CSV → select power_bi_master.csv


,CLIENTNUM,Attrition_Flag,observed_churn,churn_probability,predicted_churn,is_high_risk,risk_band,Gender,Education_Level,Marital_Status,...,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Trans_Amt,Total_Trans_Ct,Total_Amt_Chng_Q4_Q1,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Total_Relationship_Count,decision_threshold_used,model_name
0,768805383,Existing Customer,0,0.000015,0,False,Low,M,High School,Married,...,777,11914.0,1144,42,1.335,1.625,0.061,5,0.5,lightgbm
1,818770008,Existing Customer,0,0.000019,0,False,Low,F,Graduate,Single,...,864,7392.0,1291,33,1.541,3.714,0.105,6,0.5,lightgbm
2,713982108,Existing Customer,0,0.000033,0,False,Low,M,Graduate,Married,...,0,3418.0,1887,20,2.594,2.333,0.000,4,0.5,lightgbm
3,769911858,Existing Customer,0,0.000470,0,False,Low,F,High School,Unknown,...,2517,796.0,1171,20,1.405,2.333,0.760,3,0.5,lightgbm
4,709106358,Existing Customer,0,0.207069,0,False,Low,M,Uneducated,Married,...,0,4716.0,816,28,2.175,2.500,0.000,5,0.5,lightgbm


In [9]:
# =============================================================================
# 7. Figures for BI dashboards
# =============================================================================
# --- churn_overview.png (observed vs model flag) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

observed_counts = df[TARGET_COL].value_counts().reindex(
    ["Existing Customer", "Attrited Customer"]
)
axes[0].bar(
    observed_counts.index.astype(str),
    observed_counts.values,
    color=[NON_CHURN_COLOR, CHURN_COLOR],
)
axes[0].set_title("Observed attrition (historical)")
axes[0].set_ylabel("Customers")
axes[0].tick_params(axis="x", rotation=20)

risk_counts = df["is_high_risk"].value_counts().sort_index()
risk_labels = ["Not high risk", "High risk (model)"]
axes[1].bar(
    risk_labels,
    [risk_counts.get(False, 0), risk_counts.get(True, 0)],
    color=[NON_CHURN_COLOR, CHURN_COLOR],
)
axes[1].set_title(f"Model high-risk flag (threshold {HIGH_RISK_THRESHOLD:.2f})")
axes[1].set_ylabel("Customers")

fig.suptitle("Portfolio churn overview", fontsize=14, y=1.02)
overview_path = FIG_DIR / "churn_overview.png"
fig.savefig(overview_path)
plt.close(fig)
print("Saved:", overview_path)

# --- churn_by_segment.png (top segments by observed churn rate) ---
plot_df = churn_by_segment.copy()
plot_df["segment_label"] = plot_df["segment_dimension"] + ": " + plot_df["segment_value"].astype(str)
top_segments = plot_df.sort_values("observed_churn_rate", ascending=False).head(12)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=top_segments,
    y="segment_label",
    x="observed_churn_rate_pct",
    color=CHURN_COLOR,
    ax=ax,
)
ax.set_xlabel("Observed churn rate (%)")
ax.set_ylabel("")
ax.set_title("Highest observed churn segments (top 12)")
segment_fig_path = FIG_DIR / "churn_by_segment.png"
fig.savefig(segment_fig_path)
plt.close(fig)
print("Saved:", segment_fig_path)

# --- churn_model_risk_distribution.png ---
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df["churn_probability"], bins=30, color=CHURN_COLOR, ax=ax)
ax.axvline(HIGH_RISK_THRESHOLD, color="red", linestyle="--", label="High-risk threshold")
ax.set_xlabel("Predicted churn probability")
ax.set_ylabel("Customers")
ax.set_title("Model churn risk distribution")
ax.legend()
risk_dist_path = FIG_DIR / "churn_model_risk_distribution.png"
fig.savefig(risk_dist_path)
plt.close(fig)
print("Saved:", risk_dist_path)


Saved: C:\Users\Windows\Documents\Repo Folder\bank-churners-cursor-bootcamp\reports\figures\churn_overview.png


Saved: C:\Users\Windows\Documents\Repo Folder\bank-churners-cursor-bootcamp\reports\figures\churn_by_segment.png


Saved: C:\Users\Windows\Documents\Repo Folder\bank-churners-cursor-bootcamp\reports\figures\churn_model_risk_distribution.png


In [10]:
# =============================================================================
# 8. Executive summary (business narrative)
# =============================================================================
top_observed = (
    churn_by_segment.sort_values("observed_churn_rate", ascending=False)
    .head(3)[["segment_dimension", "segment_value", "observed_churn_rate_pct", "customer_count"]]
)
top_model = (
    churn_by_segment.sort_values("avg_churn_probability", ascending=False)
    .head(3)[["segment_dimension", "segment_value", "avg_churn_probability_pct", "customer_count"]]
)

def _fmt_segment(row: pd.Series) -> str:
    return f"{row['segment_dimension']} = {row['segment_value']} ({row['customer_count']} customers)"

observed_lines = "\n".join(
    f"- {_fmt_segment(r)}: **{r['observed_churn_rate_pct']:.1f}%** observed churn"
    for _, r in top_observed.iterrows()
)
model_lines = "\n".join(
    f"- {_fmt_segment(r)}: **{r['avg_churn_probability_pct']:.1f}%** average predicted risk"
    for _, r in top_model.iterrows()
)

if MODEL_METRICS_PATH.exists() and not model_metrics_summary.empty:
    metric_row = model_metrics_summary.iloc[0]
    perf_line = (
        f"Holdout test metrics for **{metric_row.get('model_name', 'final model')}**: "
        f"ROC-AUC {metric_row.get('roc_auc', float('nan')):.3f}, "
        f"recall {metric_row.get('recall_churn_class', float('nan')):.3f}, "
        f"precision {metric_row.get('precision_churn_class', float('nan')):.3f}."
    )
else:
    perf_line = "Model holdout metrics were not found; add `reports/tables/modeling_final_metrics.csv` after modeling."

summary_md = f'''# Executive Summary — Bank Customer Churn

## Business objective
Reduce preventable attrition by prioritizing retention actions on customers with the highest predicted churn risk and the segments with the highest observed churn.

## Key churn insights (observed)
- Portfolio size: **{total_customers:,}** customers.
- Observed churn rate: **{observed_churn_rate * 100:.2f}%** ({observed_attrited:,} attrited customers).
- Highest observed churn segments:
{observed_lines}

## High-risk segments and customers (model-based)
- High-risk threshold: **{HIGH_RISK_THRESHOLD:.3f}** predicted churn probability.
- Customers flagged for follow-up: **{predicted_high_risk:,}** ({predicted_high_risk_rate * 100:.2f}% of portfolio).
- Segments with highest average predicted risk:
{model_lines}
- **Power BI (single file):** import `reports/tables/power_bi_master.csv` (one row per customer with observed churn, scores, and segment bands).

## Model performance summary
{perf_line}

## Assumptions and limitations
- Observed churn reflects historical labels; it is not proof of future behavior.
- Model scores rank relative risk; they should be combined with policy and capacity constraints.
- Segment bands (age, tenure, inactivity) use fixed business bins defined in this notebook.
- Naive Bayes leakage columns are excluded from scoring, consistent with EDA and modeling rules.
- Patterns are associative, not causal.

## Recommended next actions
1. Route `high_risk_customers.csv` to retention teams for proactive outreach.
2. Design offers for the top observed-churn segments while monitoring contact fatigue.
3. Review threshold trade-offs (precision vs recall) with stakeholders quarterly.
4. Refresh scores when new customers or retrained models are available.
'''

EXEC_SUMMARY_PATH.write_text(summary_md, encoding="utf-8")
print("Saved:", EXEC_SUMMARY_PATH)


Saved: C:\Users\Windows\Documents\Repo Folder\bank-churners-cursor-bootcamp\reports\executive_summary.md


## Deliverables checklist

| Output | Path |
|--------|------|
| Churn KPI summary | `reports/tables/churn_rate_summary.csv` |
| Segment churn | `reports/tables/churn_by_segment.csv` |
| Model metrics | `reports/tables/model_metrics_summary.csv` |
| High-risk list | `reports/tables/high_risk_customers.csv` |
| **Power BI master (1 file)** | `reports/tables/power_bi_master.csv` |
| Overview figure | `reports/figures/churn_overview.png` |
| Segment figure | `reports/figures/churn_by_segment.png` |
| Risk distribution | `reports/figures/churn_model_risk_distribution.png` |
| Executive summary | `reports/executive_summary.md` |
